# 🧠 S3 — 듀얼스트림 BCI 모델 학습 (LOSO)
## Google Colab Pro+ (A100) 버전 — 담당자 A · baseline_v4

---

| 항목 | 값 |
|---|---|
| **런타임** | GPU → A100, RAM 52GB |
| **입력 데이터** | S2 전처리 결과 HDF5 (preprocessed/member_A/) |
| **EEG 스트림** | EEGNet CNN (F1=8, D=2, kern=256) → h_EEG (256dim) |
| **sEMG 스트림** | BiLSTM (hidden=128, 2layer, 양방향, LayerNorm) → h_EMG (256dim) |
| **융합** | Softmax Attention Fusion |
| **분류** | 256→128→2, ELU, Dropout(0.3) |
| **학습** | Adam lr=1e-3 · batch=32 · epoch=200 · patience=20 |
| **검증** | LOSO-CV (52명) · val_F1 macro 모니터링 |

> ### ✅ 실행 전 체크리스트
> 1. **런타임 → 런타임 유형 변경 → GPU: A100** (또는 L4) 선택
> 2. S2 전처리 결과가 `BCI_Research/preprocessed/member_A/` 에 있는지 확인
> 3. **셀을 위에서 아래로 순서대로** 실행

In [ ]:
import subprocess, os, psutil, torch

print('='*60)
print('  S3 실행 환경 확인')
print('='*60)

gpu = subprocess.run(
    ['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode == 0:
    gpu_info = gpu.stdout.strip()
    print(f'GPU  : {gpu_info}')
    if 'A100' in gpu_info or 'L4' in gpu_info:
        print('   ✅ Pro+ GPU 감지 — n_jobs=4 가능')
    else:
        print('   ⚠️  A100 권장 (LOSO 52-fold × 200 epoch)')
else:
    print('   ❌ GPU 없음 — 런타임에서 A100 선택 필요')

ram = psutil.virtual_memory()
print(f'RAM  : {ram.total/1e9:.1f} GB (여유 {ram.available/1e9:.1f} GB)')

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    DEVICE = 'cuda'
    print(f'CUDA : {torch.cuda.get_device_name(0)} | TF32 ON')
else:
    DEVICE = 'cpu'
    print('⚠️  CUDA 없음 — CPU 모드 (매우 느림)')

print('='*60)
print(f'Device: {DEVICE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
print('✅ Drive 마운트 완료')

In [ ]:
%pip install h5py numpy pandas scikit-learn tqdm -q
print('✅ 패키지 설치 완료')

In [ ]:
import os, json, time, traceback
import numpy as np
import pandas as pd
import h5py
from datetime import datetime
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset

from sklearn.metrics import f1_score, cohen_kappa_score, confusion_matrix

# 재현성 고정
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random; random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print('✅ 임포트 완료')

---
## STEP 1 — CONFIG

In [ ]:
CONFIG = {
    # ── 공통 식별 ────────────────────────────────────────────────
    'member':       'A',
    'strategy':     'baseline_v4',
    'random_seed':  42,

    # ── 경로 ─────────────────────────────────────────────────────
    'data_dir':     f'{DRIVE_ROOT}/preprocessed/member_A',
    'results_dir':  f'{DRIVE_ROOT}/results',
    'ckpt_dir':     f'{DRIVE_ROOT}/results/checkpoints_A',

    # ── 데이터 차원 (S2 baseline_v4 기준) ────────────────────────
    'n_subjects':   52,
    'n_eeg_ch':     64,
    'n_emg_ch':     4,
    # epoch_tmin=-0.5, epoch_tmax=4.0, fs=512 → n_times=2304
    'n_times':      2304,
    'n_classes':    2,          # 0=Left MI, 1=Right MI

    # ── EMG LSTM 입력 다운샘플링 ──────────────────────────────────
    # RMS 포락선 후 MI 신호 대역폭 ~5Hz → 64Hz 재샘플링도 Nyquist 기준 10배 이상
    # seq: 2304 → 288 (×8) → BiLSTM 처리 시간 8× 단축, 정보 손실 없음
    'emg_ds_factor': 8,         # 512Hz → 64Hz (stride 다운샘플)

    # ── EEGNet 파라미터 ───────────────────────────────────────────
    'eegnet_F1':        8,
    'eegnet_D':         2,      # Depthwise 배수 → F2 = F1*D = 16
    'eegnet_kern_len':  256,    # 0.5s × 512Hz = 256 샘플
    'eegnet_dropout':   0.5,

    # ── BiLSTM 파라미터 ──────────────────────────────────────────
    'lstm_hidden':  128,
    'lstm_layers':  2,
    'lstm_dropout': 0.3,

    # ── 분류 헤더 ────────────────────────────────────────────────
    'clf_dropout':  0.3,
    'feat_dim':     256,        # EEGNet FC 출력 = BiLSTM FC 출력

    # ── 학습 ─────────────────────────────────────────────────────
    'lr':           1e-3,
    'weight_decay': 1e-4,       # L2 정규화 λ
    'batch_size':   32,
    'n_epochs':     200,
    'patience':     20,

    # ── AMP (자동 혼합 정밀도) ────────────────────────────────────
    # FP16 forward + FP32 파라미터 + GradScaler → 수렴 동일, ~1.5× 속도 향상
    'use_amp':      True,       # L4/A100: True 권장 / CPU: 자동 비활성화

    # ── 실행 환경 ─────────────────────────────────────────────────
    'num_workers':  4,          # A100/L4: 4 / T4: 0
    'pin_memory':   True,
}

os.makedirs(CONFIG['ckpt_dir'], exist_ok=True)

# emg_ds_factor 적용 후 실제 EMG 시퀀스 길이 계산
CONFIG['n_times_emg'] = CONFIG['n_times'] // CONFIG['emg_ds_factor']

print('CONFIG 요약:')
for k in ['member','strategy','n_subjects','n_eeg_ch','n_emg_ch','n_times',
          'emg_ds_factor','n_times_emg',
          'eegnet_F1','eegnet_D','eegnet_kern_len','lstm_hidden','lstm_layers',
          'lr','weight_decay','batch_size','n_epochs','patience','use_amp']:
    print(f'  {k:<22}: {CONFIG[k]}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ⚡ 커널 재시작 후 빠른 복원 셀  (OPTIONAL — 선택 실행)
# ───────────────────────────────────────────────────────────────────
# ■ 최소 실행 순서 (커널 재시작 후):
#   셀 2 (GPU) → 셀 3 (Drive) → 셀 4 (pip) → 셀 5 (임포트)
#   → 셀 7 (CONFIG) → 모델 정의 셀들(9~17) → ★이 셀★
#   → 원하는 분석/학습 셀로 바로 이동
# ═══════════════════════════════════════════════════════════════════

import copy, ast
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

RESULTS_DIR = CONFIG['results_dir']

# ── CSV 로드 헬퍼 ─────────────────────────────────────────────────
def _load_csv(path, label):
    """CSV 로드 + 'sid' 컬럼 보정 + 상태 출력."""
    if not os.path.exists(path):
        print(f'⚠️  {label} CSV 없음: {os.path.basename(path)}')
        return None
    df = pd.read_csv(path)
    # 'sid' 컬럼이 없으면 첫 번째 컬럼을 sid로 간주 (방어 처리)
    if 'sid' not in df.columns:
        print(f'  ⚠️  {label}: "sid" 컬럼 없음 — 컬럼 목록: {list(df.columns)[:6]}')
        if len(df.columns) > 0:
            df = df.rename(columns={df.columns[0]: 'sid'})
            print(f'  → 첫 번째 컬럼을 "sid"로 변경')
    return df

# ── get_biased 헬퍼 ────────────────────────────────────────────────
def get_biased(ok_df, thresh=0.3):
    biased = []
    for _, row in ok_df.iterrows():
        try:
            cm = np.array(ast.literal_eval(str(row['cm'])))
            if cm.shape != (2, 2): continue
            l_rec = cm[0,0]/cm[0].sum() if cm[0].sum() > 0 else 0
            r_rec = cm[1,1]/cm[1].sum() if cm[1].sum() > 0 else 0
            if abs(l_rec - r_rec) > thresh:
                biased.append({'sid': int(row['sid']), 'l': l_rec, 'r': r_rec})
        except Exception:
            pass
    return biased

# ── 1. v4 LOSO 결과 복원 ─────────────────────────────────────────
# ★ run_loso 는 strategy 없이 'loso_progress_member_A.csv' 로 저장함
_v4_csv = os.path.join(RESULTS_DIR, f'loso_progress_member_{CONFIG["member"]}.csv')
_df = _load_csv(_v4_csv, 'v4')
if _df is not None:
    LOSO_RESULTS   = _df.to_dict('records')
    ok_v4_all      = _df[_df['status'].isin(['OK', 'RESUMED'])].copy()
    biased_v4      = get_biased(ok_v4_all)
    biased_v4_sids = {b['sid'] for b in biased_v4}
    print(f'✅ v4 복원: {len(ok_v4_all)}명  (편향 {len(biased_v4_sids)}명: {sorted(biased_v4_sids)})')
else:
    LOSO_RESULTS = []; ok_v4_all = pd.DataFrame(columns=['sid','status','acc','f1_macro','kappa','itr','cm'])
    biased_v4 = []; biased_v4_sids = set()

# ── 2. v5 LOSO 결과 복원 ─────────────────────────────────────────
_v5_csv = os.path.join(RESULTS_DIR, f'loso_progress_member_{CONFIG["member"]}_baseline_v5.csv')
_df5 = _load_csv(_v5_csv, 'v5')
if _df5 is not None:
    LOSO_RESULTS_V5 = _df5.to_dict('records')
    ok_v5           = _df5[_df5['status'].isin(['OK', 'RESUMED'])].copy()
    biased_v5       = get_biased(ok_v5)
    biased_v5_sids  = {b['sid'] for b in biased_v5}
    common_sids     = set(ok_v5['sid'].astype(int)) & set(ok_v4_all['sid'].astype(int))
    ok_v4           = ok_v4_all[ok_v4_all['sid'].astype(int).isin(common_sids)].copy()
    resolved        = biased_v4_sids - biased_v5_sids
    still           = biased_v4_sids & biased_v5_sids
    print(f'✅ v5 복원: {len(ok_v5)}명  (편향 {len(biased_v5_sids)}명)  해소: {len(resolved)}명')
else:
    LOSO_RESULTS_V5 = []; ok_v5 = pd.DataFrame(columns=['sid','status','acc','f1_macro','kappa','itr','cm'])
    biased_v5 = []; biased_v5_sids = set(); common_sids = set()
    ok_v4 = ok_v4_all.copy(); resolved = set(); still = set()

# ── 3. CONFIG_V5 / BIASED_SIDS / target_datasets 재구성 ──────────
CONFIG_V5 = copy.deepcopy(CONFIG)
CONFIG_V5.update({
    'strategy':        'baseline_v5',
    'label_smoothing': 0.1,
    'ckpt_dir':        f'{DRIVE_ROOT}/results/checkpoints_A_v5',
})

BIASED_SIDS   = [1, 5, 7, 11, 12, 15, 22, 24, 34, 36]
ABLATION_ONLY = True

if 'ALL_DATASETS' not in dir():
    ALL_DATASETS    = {}
    target_datasets = {}
    print('⚠️  ALL_DATASETS 없음 — 학습 셀 실행 전 데이터 로딩 셀(21번) 실행 필요')
else:
    target_datasets = {sid: ALL_DATASETS[sid] for sid in BIASED_SIDS if sid in ALL_DATASETS}
    print(f'✅ ALL_DATASETS 있음 — target_datasets {len(target_datasets)}명')

# ── 4. ε sweep 결과 복원 (SWEEP_RESULTS) ─────────────────────────
SWEEP_RESULTS  = {}
SWEEP_EPSILONS = [0.05]
for _eps in SWEEP_EPSILONS:
    _tag = f'v5_eps{str(_eps).replace(".", "")}'
    _csv = os.path.join(RESULTS_DIR, f'loso_progress_member_{CONFIG["member"]}_baseline_{_tag}.csv')
    _dfe = _load_csv(_csv, f'ε={_eps}')
    if _dfe is not None:
        SWEEP_RESULTS[_eps] = _dfe.to_dict('records')
        print(f'✅ ε={_eps} sweep 복원: {len(_dfe)}명')

# ── 5. class-weight sweep 결과 복원 (CW_SWEEP_RESULTS) ───────────
CW_SWEEP_RESULTS = {}
WEIGHT_SWEEP = [[1.3, 0.7]]
for _cw in WEIGHT_SWEEP:
    _w_l, _w_r = _cw
    _tag = f'v6_w{str(_w_l).replace(".", "")}_{str(_w_r).replace(".", "")}'
    _csv = os.path.join(RESULTS_DIR, f'loso_progress_member_{CONFIG["member"]}_baseline_{_tag}.csv')
    _dfc = _load_csv(_csv, f'v6 [{_w_l},{_w_r}]')
    if _dfc is not None:
        CW_SWEEP_RESULTS[tuple(_cw)] = _dfc.to_dict('records')
        print(f'✅ v6 w=[{_w_l},{_w_r}] 복원: {len(_dfc)}명')

# ── 6. 분석용 파생 변수 재구성 ────────────────────────────────────
ref_v4 = ok_v4_all[ok_v4_all['sid'].astype(int).isin(set(BIASED_SIDS))].copy() \
         if 'sid' in ok_v4_all.columns else pd.DataFrame()
ref_biased_sids = {b['sid'] for b in get_biased(ref_v4)} if not ref_v4.empty else set()

sweep_dfs = {}
for _eps, _res in SWEEP_RESULTS.items():
    _df_tmp = pd.DataFrame(_res)
    sweep_dfs[_eps] = _df_tmp[_df_tmp['status'].isin(['OK', 'RESUMED'])].copy()

cw_dfs = {}
for _cw_key, _res in CW_SWEEP_RESULTS.items():
    _df_tmp = pd.DataFrame(_res)
    cw_dfs[_cw_key] = _df_tmp[_df_tmp['status'].isin(['OK', 'RESUMED'])].copy()

print('\n' + '='*55)
print('⚡ 빠른 복원 완료')
print(f'  v4  : {len(ok_v4_all)}명 로드됨')
print(f'  v5  : {len(ok_v5)}명 로드됨')
print(f'  ε sweep : {list(SWEEP_RESULTS.keys())}')
print(f'  CW sweep: {list(CW_SWEEP_RESULTS.keys())}')
print('  ✅ 분석 셀로 바로 이동 가능')
print('  ⚠️  학습 셀 실행 전: 데이터 로딩 셀(21번) 먼저 실행 필요')
print('='*55)


---
## STEP 2 — 데이터 로딩 (HDF5 → Dataset)

In [ ]:
class BCIEpochDataset(Dataset):
    """
    S2에서 저장한 HDF5 파일 1개(피험자 1명)를 로드하는 Dataset.
    EEG: (n_epochs, n_eeg_ch, n_times)      — eeg/epochs 사용
    EMG: (n_epochs, n_emg_ch, n_times_emg) — emg/epochs에서 ds_factor stride 적용
    Labels: 1/2 → 0/1 (0=Left MI, 1=Right MI)
    """
    def __init__(self, h5_path: str, cfg: dict):
        ds = cfg.get('emg_ds_factor', 1)

        with h5py.File(h5_path, 'r') as f:
            eeg = f['eeg/epochs'][:]                           # (N, 64, T)
            lbl = f['labels'][:].astype(np.int64)              # (N,)  values 1 or 2
            if 'emg' in f and 'epochs' in f['emg']:
                emg = f['emg/epochs'][:]                       # (N, 4, T)
            else:
                emg = np.zeros(
                    (eeg.shape[0], cfg['n_emg_ch'], cfg['n_times']),
                    dtype=np.float32,
                )

        # EMG 다운샘플링: RMS 포락선은 이미 저주파 → stride 적용, 정보 손실 없음
        if ds > 1:
            emg = emg[:, :, ::ds]                              # (N, 4, T//ds)

        # 레이블 0-인덱싱 (1→0, 2→1)
        lbl = lbl - 1

        # 길이 정합
        n = min(eeg.shape[0], emg.shape[0])
        self.eeg = torch.tensor(eeg[:n], dtype=torch.float32)
        self.emg = torch.tensor(emg[:n], dtype=torch.float32)
        self.lbl = torch.tensor(lbl[:n], dtype=torch.long)

    def __len__(self):
        return len(self.lbl)

    def __getitem__(self, idx):
        return self.eeg[idx], self.emg[idx], self.lbl[idx]


def load_all_subjects(cfg: dict) -> dict:
    """52명 HDF5를 모두 로드하여 {sid: BCIEpochDataset} 반환."""
    datasets = {}
    missing = []
    for sid in range(1, cfg['n_subjects'] + 1):
        h5p = os.path.join(cfg['data_dir'], f'sub-{sid:02d}_member_{cfg["member"]}.h5')
        if not os.path.exists(h5p):
            missing.append(sid)
            continue
        try:
            datasets[sid] = BCIEpochDataset(h5p, cfg)
        except Exception:
            traceback.print_exc()
            missing.append(sid)
    print(f'✅ 로드 성공: {len(datasets)}명  |  누락: {missing if missing else "없음"}')
    if datasets:
        sid0 = next(iter(datasets))
        ds0  = datasets[sid0]
        print(f'   EEG shape : {tuple(ds0.eeg.shape)}')
        print(f'   EMG shape : {tuple(ds0.emg.shape)}  (ds×{cfg.get("emg_ds_factor",1)})')
        print(f'   Labels    : {tuple(ds0.lbl.shape)}')
    return datasets


ALL_DATASETS = load_all_subjects(CONFIG)
AVAILABLE_SIDS = sorted(ALL_DATASETS.keys())


---
## STEP 3 — 모델 정의
### 구조: EEGNet CNN + sEMG BiLSTM + Softmax Attention Fusion

In [ ]:
class EEGNetEncoder(nn.Module):
    """
    EEGNet 기반 EEG 인코더.
    Input : (batch, n_ch, n_times)
    Output: h_EEG (batch, feat_dim)
    """
    def __init__(self, n_ch: int, n_times: int, F1=8, D=2,
                 kern_len=256, dropout=0.5, feat_dim=256):
        super().__init__()
        F2 = F1 * D

        # Block 1: Temporal Convolution
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kern_len), padding=(0, kern_len // 2), bias=False),
            nn.BatchNorm2d(F1),
        )

        # Block 2: Depthwise Spatial Convolution
        self.block2 = nn.Sequential(
            nn.Conv2d(F1, F2, (n_ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 3: Separable Convolution
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, 1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        # 실제 flatten 크기를 동적으로 계산
        flat = self._get_flat_size(n_ch, n_times)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, feat_dim),
            nn.ELU(),
        )

    def _get_flat_size(self, n_ch: int, n_times: int) -> int:
        with torch.no_grad():
            x = torch.zeros(1, 1, n_ch, n_times)
            x = self.block1(x)
            x = self.block2(x)
            x = self.block3(x)
            return x.numel()

    def forward(self, x):
        # x: (batch, n_ch, n_times) → (batch, 1, n_ch, n_times)
        x = x.unsqueeze(1)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.fc(x)   # (batch, feat_dim)


# 단위 테스트
with torch.no_grad():
    _enc = EEGNetEncoder(n_ch=64, n_times=2304, feat_dim=256)
    _out = _enc(torch.zeros(4, 64, 2304))
    print(f'EEGNetEncoder 출력: {_out.shape}  (기대: (4, 256))')

In [ ]:
class EMGBiLSTMEncoder(nn.Module):
    """
    sEMG BiLSTM 인코더.
    Input : (batch, n_ch, n_times)
    Output: h_EMG (batch, feat_dim)
    """
    def __init__(self, n_ch=4, hidden=128, n_layers=2, dropout=0.3, feat_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_ch,
            hidden_size=hidden,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(hidden * 2)
        self.fc   = nn.Sequential(
            nn.Linear(hidden * 2, feat_dim),
            nn.ELU(),
        )

    def forward(self, x):
        # x: (batch, n_ch, n_times) → (batch, n_times, n_ch)
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        h = out[:, -1, :]       # 마지막 타임스텝 hidden state (batch, hidden*2)
        h = self.norm(h)
        return self.fc(h)       # (batch, feat_dim)


# 단위 테스트
with torch.no_grad():
    _enc = EMGBiLSTMEncoder(n_ch=4, hidden=128, n_layers=2, feat_dim=256)
    _out = _enc(torch.zeros(4, 4, 2304))
    print(f'EMGBiLSTMEncoder 출력: {_out.shape}  (기대: (4, 256))')

In [ ]:
class SoftmaxAttentionFusion(nn.Module):
    """
    F_fused = w_EEG * W_EEG(h_EEG) + w_EMG * W_EMG(h_EMG)
    w = softmax(attn(concat(h_EEG, h_EMG)))  → 합=1
    """
    def __init__(self, feat_dim=256):
        super().__init__()
        self.W_eeg = nn.Linear(feat_dim, feat_dim)
        self.W_emg = nn.Linear(feat_dim, feat_dim)
        self.attn  = nn.Linear(feat_dim * 2, 2)

    def forward(self, h_eeg, h_emg):
        w = F.softmax(self.attn(torch.cat([h_eeg, h_emg], dim=-1)), dim=-1)  # (B, 2)
        fused = w[:, 0:1] * self.W_eeg(h_eeg) + w[:, 1:2] * self.W_emg(h_emg)
        return fused, w


class HybridBCIModel(nn.Module):
    """
    EEG-sEMG 듀얼스트림 BCI 분류 모델.
    """
    def __init__(self, cfg: dict):
        super().__init__()
        fd = cfg['feat_dim']

        self.eeg_enc = EEGNetEncoder(
            n_ch     = cfg['n_eeg_ch'],
            n_times  = cfg['n_times'],
            F1       = cfg['eegnet_F1'],
            D        = cfg['eegnet_D'],
            kern_len = cfg['eegnet_kern_len'],
            dropout  = cfg['eegnet_dropout'],
            feat_dim = fd,
        )
        self.emg_enc = EMGBiLSTMEncoder(
            n_ch     = cfg['n_emg_ch'],
            hidden   = cfg['lstm_hidden'],
            n_layers = cfg['lstm_layers'],
            dropout  = cfg['lstm_dropout'],
            feat_dim = fd,
        )
        self.fusion = SoftmaxAttentionFusion(fd)
        self.clf = nn.Sequential(
            nn.Linear(fd, 128),
            nn.ELU(),
            nn.Dropout(cfg['clf_dropout']),
            nn.Linear(128, cfg['n_classes']),
        )

    def forward(self, eeg, emg):
        h_eeg = self.eeg_enc(eeg)                        # (B, feat_dim)
        h_emg = self.emg_enc(emg)                        # (B, feat_dim)
        fused, attn_w = self.fusion(h_eeg, h_emg)        # (B, feat_dim)
        logits = self.clf(fused)                         # (B, n_classes)
        return logits, attn_w


# 전체 모델 단위 테스트
with torch.no_grad():
    _m = HybridBCIModel(CONFIG)
    _logits, _w = _m(torch.zeros(4, 64, 2304), torch.zeros(4, 4, 2304))
    total_p = sum(p.numel() for p in _m.parameters())
    train_p = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f'HybridBCIModel 출력 — logits: {_logits.shape}  attn_w: {_w.shape}')
    print(f'파라미터 수: 전체={total_p:,}  학습가능={train_p:,}')

---
## STEP 4 — 학습 / 평가 함수

In [ ]:
def compute_itr(n_classes: int, accuracy: float, trial_sec: float) -> float:
    """
    ITR (bits/min) 계산.
    accuracy가 1/n 이하이면 0 반환 (음수 방지).
    """
    p = np.clip(accuracy, 1e-6, 1.0 - 1e-6)
    n = n_classes
    if p <= 1.0 / n:
        return 0.0
    b = np.log2(n) + p * np.log2(p) + (1 - p) * np.log2((1 - p) / max(n - 1, 1))
    return b * (60.0 / trial_sec)


def evaluate_predictions(y_true, y_pred, n_classes=2, trial_sec=4.0):
    """F1-macro, Cohen's κ, ITR 계산."""
    acc  = (y_true == y_pred).mean()
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    kap  = cohen_kappa_score(y_true, y_pred)
    itr  = compute_itr(n_classes, acc, trial_sec)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(n_classes)))
    return {'acc': float(acc), 'f1_macro': float(f1), 'kappa': float(kap),
            'itr': float(itr), 'cm': cm.tolist()}


print('✅ 평가 함수 정의 완료')

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device, scaler):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    use_amp = scaler is not None

    for eeg, emg, lbl in loader:
        eeg, emg, lbl = eeg.to(device), emg.to(device), lbl.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits, _ = model(eeg, emg)
            loss = criterion(logits, lbl)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * len(lbl)
        correct    += (logits.argmax(1) == lbl).sum().item()
        n          += len(lbl)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, scaler):
    model.eval()
    total_loss, preds, trues = 0.0, [], []
    use_amp = scaler is not None

    for eeg, emg, lbl in loader:
        eeg, emg, lbl = eeg.to(device), emg.to(device), lbl.to(device)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits, _ = model(eeg, emg)
            total_loss += criterion(logits, lbl).item() * len(lbl)
        preds.extend(logits.argmax(1).cpu().numpy())
        trues.extend(lbl.cpu().numpy())

    n  = len(trues)
    f1 = f1_score(trues, preds, average='macro', zero_division=0)
    return total_loss / n, f1, np.array(preds), np.array(trues)


print('✅ 학습/평가 함수 정의 완료 (AMP 지원)')


---
## STEP 5 — LOSO 교차검증 루프

In [ ]:
def run_loso_fold(test_sid: int, all_datasets: dict, cfg: dict, device: str) -> dict:
    """
    단일 LOSO fold.
    - 체크포인트(best_s{sid}.pt)가 이미 존재하면 재학습 없이 평가만 수행.
    - 체크포인트가 없으면 처음부터 학습.
    """
    ckpt_path = os.path.join(cfg['ckpt_dir'], f'best_s{test_sid:02d}.pt')
    test_ds   = all_datasets[test_sid]
    test_loader = DataLoader(
        test_ds, batch_size=cfg['batch_size'], shuffle=False,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
    )
    use_amp = cfg.get('use_amp', False) and device != 'cpu'
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ── 체크포인트 복원 모드 ──────────────────────────────────────
    if os.path.exists(ckpt_path):
        torch.manual_seed(cfg['random_seed'])
        model = HybridBCIModel(cfg).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        criterion = nn.CrossEntropyLoss()
        _, best_f1, best_preds, best_trues = eval_epoch(
            model, test_loader, criterion, device, scaler
        )
        metrics = evaluate_predictions(best_trues, best_preds,
                                       n_classes=cfg['n_classes'], trial_sec=4.0)
        return {'sid': test_sid, 'status': 'RESUMED', 'best_epoch': -1, **metrics}

    # ── 신규 학습 모드 ────────────────────────────────────────────
    train_sids = [s for s in all_datasets if s != test_sid]
    train_ds   = ConcatDataset([all_datasets[s] for s in train_sids])
    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'], shuffle=True,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
        drop_last=True,
    )

    torch.manual_seed(cfg['random_seed'])
    model     = HybridBCIModel(cfg).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )

    best_f1, best_preds, best_trues, no_improve = -1.0, None, None, 0

    for epoch in range(1, cfg['n_epochs'] + 1):
        train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        _, val_f1, preds, trues = eval_epoch(model, test_loader, criterion, device, scaler)

        if val_f1 > best_f1:
            best_f1, best_preds, best_trues, no_improve = val_f1, preds.copy(), trues.copy(), 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if no_improve >= cfg['patience']:
            print(f'    s{test_sid:02d} → Early stop ep{epoch} | best F1={best_f1:.4f}')
            break

    metrics = evaluate_predictions(best_trues, best_preds,
                                   n_classes=cfg['n_classes'], trial_sec=4.0)
    return {'sid': test_sid, 'status': 'OK', 'best_epoch': epoch - no_improve, **metrics}


def run_loso(all_datasets: dict, cfg: dict, device: str) -> list:
    """
    전체 LOSO 루프.
    - fold마다 완료 즉시 progress CSV에 저장 → 연결 끊겨도 이어받기 가능.
    - 이미 progress CSV에 기록된 sid는 건너뜀 (완전 재시작 방어).
    """
    progress_csv = os.path.join(cfg['results_dir'],
                                f'loso_progress_member_{cfg["member"]}.csv')

    # 이전 진행 결과 불러오기
    if os.path.exists(progress_csv):
        done_df  = pd.read_csv(progress_csv)
        done_ids = set(done_df['sid'].astype(int).tolist())
        results  = done_df.to_dict('records')
        print(f'📂 이전 진행 결과 로드: {len(done_ids)}개 fold 복원')
    else:
        done_ids, results = set(), []

    remaining = [s for s in sorted(all_datasets.keys()) if s not in done_ids]
    print(f'▶ 남은 fold: {len(remaining)}개 / 전체 {len(all_datasets)}개')

    for test_sid in tqdm(remaining, desc='LOSO'):
        try:
            r = run_loso_fold(test_sid, all_datasets, cfg, device)
            results.append(r)
            tag = '🔄' if r['status'] == 'RESUMED' else '✅'
            print(f'  {tag} s{test_sid:02d} [{r["status"]}] — '
                  f'acc={r["acc"]:.3f}  F1={r["f1_macro"]:.3f}  '
                  f'κ={r["kappa"]:.3f}  ITR={r["itr"]:.1f} bits/min')
        except Exception:
            traceback.print_exc()
            results.append({'sid': test_sid, 'status': 'ERROR'})
            print(f'  ❌ s{test_sid:02d}: ERROR — 건너뜀')

        # fold 완료 즉시 저장
        pd.DataFrame(results).to_csv(progress_csv, index=False)

    return results


print('✅ LOSO 함수 정의 완료 (체크포인트 복원 + 즉시 저장)')


---
## STEP 6 — 실행

In [ ]:
print(f'실행 시작: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'대상 피험자: {len(AVAILABLE_SIDS)}명  |  Device: {DEVICE}')
print(f'EEGNet kern_len={CONFIG["eegnet_kern_len"]} | LSTM hidden={CONFIG["lstm_hidden"]} | batch={CONFIG["batch_size"]}')
print('='*60)

t0 = time.time()
LOSO_RESULTS = run_loso(ALL_DATASETS, CONFIG, DEVICE)
elapsed = time.time() - t0

print(f'\n🏁 LOSO 완료 — 소요시간: {elapsed/60:.1f}분')

---
## STEP 7 — 결과 분석 및 저장

> LOSO 완료 후 아래 셀을 순서대로 실행하세요.
> - **7-A**: 기술통계 요약 + 논문 기준 등급 + CSV 저장
> - **7-B**: 피험자별 성능 분포 시각화 (막대 + Boxplot + ITR 히스토그램)
> - **7-C**: 누적 Confusion Matrix 히트맵 (절대값 + 정규화)
> - **7-D**: 하위/상위 피험자 + Right MI 편향 피험자 목록
> - **7-E**: Right MI 편향 원인 진단 (데이터 불균형 vs 모델 과적합)

In [ ]:
# ── STEP 7-A: 기술통계 요약 + CSV 저장 ────────────────────────────
df = pd.DataFrame(LOSO_RESULTS)

# RESUMED + OK 모두 성공 fold로 집계
ok  = df[df['status'].isin(['OK', 'RESUMED'])].copy()
err = df[df['status'] == 'ERROR']

print('='*60)
print(f'  LOSO 결과 요약 — {CONFIG["strategy"]} (member {CONFIG["member"]})')
print('='*60)

metrics_labels = [
    ('acc',       'Accuracy       '),
    ('f1_macro',  'F1-macro       '),
    ('kappa',     "Cohen's κ      "),
    ('itr',       'ITR (bits/min) '),
]

if len(ok) > 0:
    for col, label in metrics_labels:
        v = ok[col]
        print(f'  {label}: {v.mean():.4f} ± {v.std():.4f}'
              f'  [min {v.min():.4f} / max {v.max():.4f}]')
    print(f'\n  성공 fold : {len(ok)} / {len(df)}  (RESUMED: {(ok["status"]=="RESUMED").sum()}, 신규학습: {(ok["status"]=="OK").sum()})')
    if len(err):
        print(f'  ERROR fold: s{sorted(err["sid"].astype(int).tolist())}')
else:
    print('  결과 없음')

# ── 논문 기준 해석 ────────────────────────────────────────────────
print('\n── 논문 기준 해석 ──────────────────────────────────')
mean_acc = ok['acc'].mean() if len(ok) else 0
mean_f1  = ok['f1_macro'].mean() if len(ok) else 0
mean_kap = ok['kappa'].mean() if len(ok) else 0
mean_itr = ok['itr'].mean() if len(ok) else 0

def grade(acc):
    if acc >= 0.85: return '🟢 Excellent (state-of-the-art)'
    elif acc >= 0.75: return '🟡 Good (논문 게재 가능 수준)'
    elif acc >= 0.65: return '🟠 Fair (개선 여지 있음)'
    else: return '🔴 Poor (< 65%, 유의한 분류 어려움)'

print(f'  Accuracy {mean_acc:.1%} → {grade(mean_acc)}')
if mean_kap >= 0.8:
    kap_label = 'Almost perfect (≥0.8)'
elif mean_kap >= 0.6:
    kap_label = 'Substantial (0.6~0.8)'
elif mean_kap >= 0.4:
    kap_label = 'Moderate (0.4~0.6)'
else:
    kap_label = 'Fair (<0.4)'
print(f'  Cohen κ  {mean_kap:.3f} → {kap_label}')

# ITR: n=2 classes, trial=4.0s → 이론 최대 = log2(2)×(60/4) = 15 bits/min
itr_max_theoretical = np.log2(CONFIG['n_classes']) * (60.0 / 4.0)
itr_practical_threshold = 5.0   # acc≈82% 수준
n_itr_ok = (ok['itr'] >= itr_practical_threshold).sum()
print(f'  ITR      {mean_itr:.2f} bits/min')
print(f'           이론 최대: {itr_max_theoretical:.1f} bits/min (n=2, trial=4s)')
print(f'           실용 기준 ≥{itr_practical_threshold} bits/min 달성: {n_itr_ok}명 ({n_itr_ok/len(ok)*100:.1f}%)')

# ── CSV 저장 ────────────────────────────────────────────────────
ts = datetime.now().strftime('%Y%m%d_%H%M')
save_path = os.path.join(CONFIG['results_dir'],
    f'loso_results_member_{CONFIG["member"]}_{CONFIG["strategy"]}_{ts}.csv')
df.to_csv(save_path, index=False)
print(f'\n✅ 저장: {save_path}')

# results.csv 누적
results_csv = os.path.join(CONFIG['results_dir'], 'results.csv')
if len(ok) > 0:
    row = pd.DataFrame([{
        'member': CONFIG['member'], 'strategy': CONFIG['strategy'], 'stage': 'S3',
        'n_folds': len(ok),
        'acc_mean': ok['acc'].mean(), 'acc_std': ok['acc'].std(),
        'f1_mean': ok['f1_macro'].mean(), 'f1_std': ok['f1_macro'].std(),
        'kappa_mean': ok['kappa'].mean(), 'itr_mean': ok['itr'].mean(),
        'itr_max_theoretical': itr_max_theoretical,
        'saved_at': datetime.now().isoformat(),
    }])
    if os.path.exists(results_csv):
        pd.concat([pd.read_csv(results_csv), row], ignore_index=True).to_csv(results_csv, index=False)
    else:
        row.to_csv(results_csv, index=False)
    print(f'✅ results.csv 업데이트 완료')


In [ ]:
# ── STEP 7-B: 피험자별 성능 분포 시각화 ──────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

ok_sorted = ok.sort_values('f1_macro').reset_index(drop=True)
n = len(ok_sorted)
sids = ok_sorted['sid'].astype(int).tolist()

# n=2 classes, trial=4.0s → 이론 최대 = 15 bits/min, 실용 기준 = 5 bits/min
itr_max_theoretical = np.log2(CONFIG['n_classes']) * (60.0 / 4.0)
itr_practical = 5.0

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'LOSO 결과 분포 — Member {CONFIG["member"]} ({CONFIG["strategy"]})',
             fontsize=14, fontweight='bold')

# ① 피험자별 Accuracy 막대그래프
colors = ['#d62728' if v < 0.6 else '#ff7f0e' if v < 0.7 else '#2ca02c'
          for v in ok_sorted['acc']]
axes[0].bar(range(n), ok_sorted['acc'], color=colors, width=0.7)
axes[0].axhline(ok_sorted['acc'].mean(), color='navy', linewidth=1.5,
                linestyle='--', label=f'mean={ok_sorted["acc"].mean():.3f}')
axes[0].axhline(0.5, color='gray', linewidth=1, linestyle=':')
axes[0].set_xticks(range(n))
axes[0].set_xticklabels([f's{s:02d}' for s in sids], rotation=90, fontsize=7)
axes[0].set_ylabel('Accuracy')
axes[0].set_title('피험자별 Accuracy')
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.05)

# ② F1 / kappa 박스플롯
data_box = [ok['acc'].values, ok['f1_macro'].values, ok['kappa'].values]
bp = axes[1].boxplot(data_box, patch_artist=True,
                     labels=['Accuracy', 'F1-macro', "Cohen's κ"],
                     medianprops=dict(color='black', linewidth=2))
colors_box = ['#aec7e8', '#ffbb78', '#98df8a']
for patch, c in zip(bp['boxes'], colors_box):
    patch.set_facecolor(c)
axes[1].set_title('성능 지표 분포')
axes[1].set_ylabel('Score')
axes[1].grid(axis='y', alpha=0.3)

# ③ ITR 분포 히스토그램 (기준선 수정: 20→5 bits/min)
axes[2].hist(ok['itr'], bins=15, color='#9467bd', edgecolor='white', alpha=0.85)
axes[2].axvline(ok['itr'].mean(), color='red', linewidth=1.5,
                linestyle='--', label=f'mean={ok["itr"].mean():.1f}')
axes[2].axvline(itr_practical, color='orange', linewidth=1.5, linestyle=':',
                label=f'{itr_practical} bits/min (실용 기준, acc≈82%)')
axes[2].axvline(itr_max_theoretical, color='gray', linewidth=1, linestyle=':',
                label=f'{itr_max_theoretical:.0f} bits/min (이론 최대)')
axes[2].set_xlabel('ITR (bits/min)')
axes[2].set_ylabel('피험자 수')
axes[2].set_title('ITR 분포')
axes[2].legend(fontsize=8)

plt.tight_layout()
plot_path = os.path.join(CONFIG['results_dir'],
    f'loso_dist_member_{CONFIG["member"]}_{datetime.now().strftime("%Y%m%d_%H%M")}.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 그래프 저장: {plot_path}')


In [ ]:
# ── STEP 7-C: Confusion Matrix 누적 히트맵 ────────────────────────
import numpy as np
import ast

# cm 컬럼이 문자열로 저장된 경우 파싱
def parse_cm(cm_val):
    if isinstance(cm_val, list):
        return np.array(cm_val)
    try:
        return np.array(ast.literal_eval(str(cm_val)))
    except Exception:
        return None

cm_total = np.zeros((CONFIG['n_classes'], CONFIG['n_classes']), dtype=int)
for _, row in ok.iterrows():
    cm = parse_cm(row.get('cm'))
    if cm is not None and cm.shape == (CONFIG['n_classes'], CONFIG['n_classes']):
        cm_total += cm

labels = ['Left MI', 'Right MI']
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f'Confusion Matrix — Member {CONFIG["member"]}', fontsize=13, fontweight='bold')

# 절대값
im0 = axes[0].imshow(cm_total, cmap='Blues')
axes[0].set_title(f'누적 CM (총 {cm_total.sum()}개 trial)')
for i in range(CONFIG['n_classes']):
    for j in range(CONFIG['n_classes']):
        axes[0].text(j, i, str(cm_total[i, j]),
                     ha='center', va='center',
                     color='white' if cm_total[i, j] > cm_total.max()*0.5 else 'black',
                     fontsize=14, fontweight='bold')
axes[0].set_xticks(range(CONFIG['n_classes'])); axes[0].set_xticklabels(labels)
axes[0].set_yticks(range(CONFIG['n_classes'])); axes[0].set_yticklabels(labels)
axes[0].set_xlabel('예측'); axes[0].set_ylabel('실제')
plt.colorbar(im0, ax=axes[0])

# 정규화 (행 기준)
cm_norm = cm_total.astype(float) / cm_total.sum(axis=1, keepdims=True)
im1 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('정규화 CM (행 합=1)')
for i in range(CONFIG['n_classes']):
    for j in range(CONFIG['n_classes']):
        axes[1].text(j, i, f'{cm_norm[i,j]:.3f}',
                     ha='center', va='center',
                     color='white' if cm_norm[i,j] > 0.5 else 'black',
                     fontsize=13, fontweight='bold')
axes[1].set_xticks(range(CONFIG['n_classes'])); axes[1].set_xticklabels(labels)
axes[1].set_yticks(range(CONFIG['n_classes'])); axes[1].set_yticklabels(labels)
axes[1].set_xlabel('예측'); axes[1].set_ylabel('실제')
plt.colorbar(im1, ax=axes[1])

# 클래스별 민감도 출력
for i, lbl in enumerate(labels):
    print(f'  {lbl} 민감도(Recall): {cm_norm[i,i]:.3f}  '
          f'(오분류→{labels[1-i]}: {cm_norm[i,1-i]:.3f})')

plt.tight_layout()
cm_path = os.path.join(CONFIG['results_dir'],
    f'cm_member_{CONFIG["member"]}_{datetime.now().strftime("%Y%m%d_%H%M")}.png')
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ CM 저장: {cm_path}')


In [ ]:
# ── STEP 7-D: 성능 하위/상위 피험자 분석 + 클래스 편향 ───────────
LOW_THRESH  = 0.60
HIGH_THRESH = 0.80
# n=2 classes, trial=4.0s → 이론 최대 ITR = 15 bits/min
# 실용 기준: ≥5 bits/min (acc≈82% 수준)
ITR_THRESH  = 5.0

low  = ok[ok['acc'] < LOW_THRESH].sort_values('acc')
high = ok[ok['acc'] >= HIGH_THRESH].sort_values('acc', ascending=False)

print('='*55)
print(f'  🔴 하위 피험자 (acc < {LOW_THRESH}) — {len(low)}명')
print('='*55)
if len(low) > 0:
    print(low[['sid','acc','f1_macro','kappa','itr']].to_string(index=False))
    print(f'\n  ⚠️  하위 비율: {len(low)/len(ok)*100:.1f}%')
    print(f'  → 개인 맞춤형 파인튜닝 또는 논문 outlier 제거 서브섹션 권장')
else:
    print('  없음 — 전원 acc ≥ 60%')

print()
print('='*55)
print(f'  🟢 상위 피험자 (acc ≥ {HIGH_THRESH}) — {len(high)}명')
print('='*55)
if len(high) > 0:
    print(high[['sid','acc','f1_macro','kappa','itr']].to_string(index=False))

# ── 클래스 편향 분석 ─────────────────────────────────────────────
print()
print('='*55)
print('  ⚠️  Right MI 편향 피험자 (|Left-Right Recall| > 0.3)')
print('='*55)
import ast, numpy as np

biased = []
for _, row in ok.iterrows():
    try:
        cm = np.array(ast.literal_eval(str(row['cm'])))
        if cm.shape != (2, 2): continue
        l_rec = cm[0,0] / cm[0].sum() if cm[0].sum() > 0 else 0
        r_rec = cm[1,1] / cm[1].sum() if cm[1].sum() > 0 else 0
        if abs(l_rec - r_rec) > 0.3:
            biased.append({'sid': int(row['sid']), 'acc': row['acc'],
                           'left_recall': l_rec, 'right_recall': r_rec,
                           'bias': 'Right' if r_rec > l_rec else 'Left'})
    except Exception:
        pass

if biased:
    print(f'  총 {len(biased)}명 편향 감지:')
    for b in sorted(biased, key=lambda x: x['acc']):
        print(f'  s{b["sid"]:02d}: acc={b["acc"]:.3f} | Left={b["left_recall"]:.2f}  Right={b["right_recall"]:.2f} → {b["bias"]} 편향')
else:
    print('  없음')

# ── ITR 요약 ─────────────────────────────────────────────────────
print()
print('='*55)
print(f'  ITR 분석 (기준: ≥{ITR_THRESH} bits/min / 이론최대: 15 bits/min)')
print('='*55)
itr_ok = ok[ok['itr'] >= ITR_THRESH]
print(f'  달성 피험자: {len(itr_ok)}명 / {len(ok)}명 ({len(itr_ok)/len(ok)*100:.1f}%)')
if len(itr_ok) > 0:
    print(itr_ok[['sid','acc','itr']].sort_values('itr', ascending=False).to_string(index=False))

print()
print('='*55)
print('  전체 피험자 성능 테이블 (acc 오름차순)')
print('='*55)
print(ok.sort_values('acc')[['sid','status','acc','f1_macro','kappa','itr']].to_string(index=False))


In [ ]:
# ── STEP 7-E: Right MI 편향 원인 진단 — 클래스 비율 확인 ──────────
# 모델 편향(학습 불균형)인지, 피험자 고유 특성인지 구분:
#   - HDF5 labels에서 피험자별 Left/Right trial 수 집계
#   - Right% ≈ 50% 인데도 편향 → 모델 과적합 (class_weight or label smoothing 필요)
#   - Right% > 55% 이면 → 데이터 자체 불균형 (class_weight 필요)

print('='*60)
print('  피험자별 클래스 비율 (Left MI : Right MI)')
print('='*60)

# 7-D에서 감지된 Right 편향 피험자 SID 목록 (자동 추출)
biased_sids = set()
for _, row in ok.iterrows():
    try:
        cm = np.array(ast.literal_eval(str(row['cm'])))
        if cm.shape != (2, 2): continue
        l_rec = cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0
        r_rec = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0
        if abs(l_rec - r_rec) > 0.3:
            biased_sids.add(int(row['sid']))
    except Exception:
        pass

print(f'  편향 감지 피험자 ({len(biased_sids)}명): {sorted(biased_sids)}')

# HDF5에서 클래스 비율 집계
ratio_records = []
for sid, ds in sorted(ALL_DATASETS.items()):
    lbls = ds.lbl.numpy()          # 0=Left MI, 1=Right MI
    n_left  = int((lbls == 0).sum())
    n_right = int((lbls == 1).sum())
    total   = len(lbls)
    right_ratio = n_right / total if total > 0 else 0.5
    ratio_records.append({
        'sid': sid, 'n_left': n_left, 'n_right': n_right,
        'total': total, 'right_ratio': right_ratio,
        'is_biased': sid in biased_sids,
    })

ratio_df = pd.DataFrame(ratio_records)

# 전체 평균 클래스 비율
avg_right = ratio_df['right_ratio'].mean()
print(f'\n  전체 평균 Right MI 비율: {avg_right:.3f}  (이상적: 0.500)')
if abs(avg_right - 0.5) > 0.05:
    print(f'  ⚠️  데이터셋 전체에 클래스 불균형 존재 (Right {avg_right:.1%})')
else:
    print(f'  ✅ 전체 클래스 균형 양호')

# 편향 피험자의 실제 클래스 비율
print('\n  [Right 편향 피험자의 실제 클래스 비율]')
print(f'  {"sid":>4}  {"Left":>5}  {"Right":>5}  {"Right%":>7}  판정')
print('  ' + '-'*52)
data_imbalanced = ratio_df[ratio_df['is_biased']].sort_values('right_ratio', ascending=False)
for _, r in data_imbalanced.iterrows():
    if r['right_ratio'] > 0.55:
        verdict = '⚠️  데이터 불균형'
    elif r['right_ratio'] < 0.45:
        verdict = '⚠️  Left 과다 (역설적)'
    else:
        verdict = '→ 모델 편향 (데이터 균형)'
    print(f'  s{int(r["sid"]):02d}   {r["n_left"]:5d}  {r["n_right"]:5d}  {r["right_ratio"]:6.1%}  {verdict}')

# 결론 도출
n_data_imbalanced = (data_imbalanced['right_ratio'] > 0.55).sum()
print()
if n_data_imbalanced >= len(biased_sids) * 0.5:
    print('  📌 결론: 데이터 불균형이 주원인')
    print('     → CrossEntropyLoss에 class_weight 추가 권장:')
    print('        w = torch.tensor([avg_right, 1-avg_right]).to(device)')
    print('        criterion = nn.CrossEntropyLoss(weight=w)')
else:
    print('  📌 결론: 데이터는 균형적 → 모델이 Right MI에 과적합')
    print('     → label smoothing(ε=0.1) 또는 focal loss 검토')

# ── 시각화 ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('클래스 비율 분석 (Left MI vs Right MI)', fontsize=12, fontweight='bold')

# 전체 분포 히스토그램
axes[0].hist(ratio_df['right_ratio'], bins=20, color='#4e79a7', edgecolor='white', alpha=0.85)
axes[0].axvline(0.5,       color='red',    linestyle='--', linewidth=1.5, label='균형 기준 (0.50)')
axes[0].axvline(avg_right, color='orange', linestyle='--', linewidth=1.5,
                label=f'전체 평균 ({avg_right:.3f})')
axes[0].set_xlabel('Right MI 비율')
axes[0].set_ylabel('피험자 수')
axes[0].set_title('전체 피험자 클래스 비율 분포')
axes[0].legend()

# 피험자별 막대 (편향 피험자 강조)
bar_colors = ['#d62728' if sid in biased_sids else '#4e79a7'
              for sid in ratio_df['sid']]
axes[1].bar(range(len(ratio_df)), ratio_df['right_ratio'], color=bar_colors, width=0.7)
axes[1].axhline(0.5,   color='red',    linestyle='--', linewidth=1.5, label='균형 기준')
axes[1].axhline(0.55,  color='orange', linestyle=':',  linewidth=1,   label='불균형 임계 (55%)')
axes[1].set_xticks(range(len(ratio_df)))
axes[1].set_xticklabels([f's{int(s):02d}' for s in ratio_df['sid']], rotation=90, fontsize=7)
axes[1].set_ylabel('Right MI 비율')
axes[1].set_title('피험자별 Right MI 비율\n(빨강 = 모델 편향 감지 피험자)')
axes[1].set_ylim(0, 1)
red_p  = mpatches.Patch(color='#d62728', label='편향 피험자')
blue_p = mpatches.Patch(color='#4e79a7', label='정상 피험자')
axes[1].legend(handles=[red_p, blue_p])

plt.tight_layout()
ratio_path = os.path.join(CONFIG['results_dir'],
    f'class_ratio_member_{CONFIG["member"]}_{datetime.now().strftime("%Y%m%d_%H%M")}.png')
plt.savefig(ratio_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 클래스 비율 그래프 저장: {ratio_path}')


---
## STEP 8 — baseline_v5: Label Smoothing (Right MI 편향 개선)

> **7-E 진단 결과**: 데이터는 균형(Right ≈ 50%)이나 모델이 Right MI에 과적합  
> **해결**: `CrossEntropyLoss(label_smoothing=0.1)` 적용 → 단 1줄 변경  
>
> - **8-A**: CONFIG_V5 정의 (strategy=`baseline_v5`, label_smoothing=0.1)  
> - **8-B**: v5 학습 함수 정의 (run_loso_fold_v5 / run_loso_v5)  
> - **8-C**: LOSO 실행 (Colab에서 실행, ~3시간)  
> - **8-D**: v4 vs v5 성능 비교


In [ ]:
# ── STEP 8-A: CONFIG_V5 — label_smoothing=0.1 추가 ───────────────
import copy

CONFIG_V5 = copy.deepcopy(CONFIG)
CONFIG_V5.update({
    'strategy':        'baseline_v5',
    'label_smoothing': 0.1,          # CrossEntropyLoss smoothing 파라미터
    'ckpt_dir':        f'{DRIVE_ROOT}/results/checkpoints_A_v5',
})
os.makedirs(CONFIG_V5['ckpt_dir'], exist_ok=True)

print('CONFIG_V5 변경 항목:')
print(f'  strategy       : {CONFIG["strategy"]} → {CONFIG_V5["strategy"]}')
print(f'  label_smoothing: (없음)        → {CONFIG_V5["label_smoothing"]}')
print(f'  ckpt_dir       : {CONFIG_V5["ckpt_dir"]}')
print('\n  그 외 모든 하이퍼파라미터 (lr, batch, epoch, 모델 구조 등) 동일')


In [ ]:
# ── STEP 8-B: v5 LOSO 함수 (label smoothing 적용) ────────────────

def run_loso_fold_v5(test_sid: int, all_datasets: dict, cfg: dict, device: str) -> dict:
    """
    baseline_v5 전용 단일 fold.
    run_loso_fold와 동일하되, criterion에 label_smoothing 적용.
    """
    ckpt_path = os.path.join(cfg['ckpt_dir'], f'best_s{test_sid:02d}.pt')
    test_ds = all_datasets[test_sid]
    test_loader = DataLoader(
        test_ds, batch_size=cfg['batch_size'], shuffle=False,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
    )
    use_amp = cfg.get('use_amp', False) and device != 'cpu'
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ★ 핵심 변경: label_smoothing 적용
    smoothing = cfg.get('label_smoothing', 0.0)
    criterion = nn.CrossEntropyLoss(label_smoothing=smoothing)

    # ── 체크포인트 복원 ───────────────────────────────────────────
    if os.path.exists(ckpt_path):
        torch.manual_seed(cfg['random_seed'])
        model = HybridBCIModel(cfg).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        _, best_f1, best_preds, best_trues = eval_epoch(
            model, test_loader, criterion, device, scaler
        )
        metrics = evaluate_predictions(best_trues, best_preds,
                                       n_classes=cfg['n_classes'], trial_sec=4.0)
        return {'sid': test_sid, 'status': 'RESUMED', 'best_epoch': -1, **metrics}

    # ── 신규 학습 ─────────────────────────────────────────────────
    train_sids = [s for s in all_datasets if s != test_sid]
    train_ds   = ConcatDataset([all_datasets[s] for s in train_sids])
    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'], shuffle=True,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
        drop_last=True,
    )

    torch.manual_seed(cfg['random_seed'])
    model     = HybridBCIModel(cfg).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )

    best_f1, best_preds, best_trues, no_improve = -1.0, None, None, 0

    for epoch in range(1, cfg['n_epochs'] + 1):
        train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        _, val_f1, preds, trues = eval_epoch(
            model, test_loader, criterion, device, scaler
        )

        if val_f1 > best_f1:
            best_f1, best_preds, best_trues, no_improve = val_f1, preds.copy(), trues.copy(), 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if no_improve >= cfg['patience']:
            print(f'    s{test_sid:02d} → Early stop ep{epoch} | best F1={best_f1:.4f}')
            break

    metrics = evaluate_predictions(best_trues, best_preds,
                                   n_classes=cfg['n_classes'], trial_sec=4.0)
    return {'sid': test_sid, 'status': 'OK', 'best_epoch': epoch - no_improve, **metrics}


def run_loso_v5(all_datasets: dict, cfg: dict, device: str) -> list:
    """
    v5 전용 LOSO 루프.
    progress CSV 이름에 strategy를 포함 → v4 CSV와 충돌 없음.
    """
    progress_csv = os.path.join(
        cfg['results_dir'],
        f'loso_progress_member_{cfg["member"]}_{cfg["strategy"]}.csv'
    )

    if os.path.exists(progress_csv):
        done_df  = pd.read_csv(progress_csv)
        done_ids = set(done_df['sid'].astype(int).tolist())
        results  = done_df.to_dict('records')
        print(f'📂 이전 진행 결과 로드: {len(done_ids)}개 fold 복원')
    else:
        done_ids, results = set(), []

    remaining = [s for s in sorted(all_datasets.keys()) if s not in done_ids]
    print(f'▶ 남은 fold: {len(remaining)}개 / 전체 {len(all_datasets)}개')
    print(f'  label_smoothing = {cfg["label_smoothing"]}')

    for test_sid in tqdm(remaining, desc=f'LOSO {cfg["strategy"]}'):
        try:
            r = run_loso_fold_v5(test_sid, all_datasets, cfg, device)
            results.append(r)
            tag = '🔄' if r['status'] == 'RESUMED' else '✅'
            print(f'  {tag} s{test_sid:02d} [{r["status"]}] — '
                  f'acc={r["acc"]:.3f}  F1={r["f1_macro"]:.3f}  '
                  f'κ={r["kappa"]:.3f}  ITR={r["itr"]:.1f} bits/min')
        except Exception:
            traceback.print_exc()
            results.append({'sid': test_sid, 'status': 'ERROR'})
            print(f'  ❌ s{test_sid:02d}: ERROR — 건너뜀')

        pd.DataFrame(results).to_csv(progress_csv, index=False)

    return results


print('✅ STEP 8-B 함수 정의 완료')
print(f'  label_smoothing = {CONFIG_V5["label_smoothing"]}  (CrossEntropyLoss)')


In [ ]:
# ── STEP 8-C: baseline_v5 LOSO 실행 ──────────────────────────────
# ⚠️  컴퓨팅 단위 절약을 위해 기본값은 '편향 피험자 10명만' 타겟 ablation
#
# [컴퓨팅 단위 추정]
#   A100 소비율 ≈ 10 CU/시간
#   전체 52-fold: v4 기준 10시간+ → ~100 CU 필요  ← 31.96 CU로 불가
#   타겟 ablation (10명): ~2시간               → ~20 CU 소비, 잔여 ~12 CU
#
# [모드 선택]
#   ABLATION_ONLY = True  → 편향 10명만 재학습 (권장, ~2시간)
#   ABLATION_ONLY = False → 전체 52명 재학습   (컴퓨팅 단위 충분할 때)

ABLATION_ONLY = True   # ← 필요 시 False로 변경

# 7-D/7-E에서 감지된 Right MI 편향 피험자 (|Left - Right Recall| > 0.3)
BIASED_SIDS = [1, 5, 7, 11, 12, 15, 22, 24, 34, 36]  # v4 결과 기준

if ABLATION_ONLY:
    target_datasets = {sid: ALL_DATASETS[sid] for sid in BIASED_SIDS
                       if sid in ALL_DATASETS}
    print(f'🎯 타겟 ablation 모드: 편향 {len(target_datasets)}명만 v5 재학습')
    print(f'   대상 SID: {sorted(target_datasets.keys())}')
    print(f'   예상 소요: ~2시간 (A100 기준) ≈ ~20 CU')
else:
    target_datasets = ALL_DATASETS
    print(f'🔄 전체 모드: 52명 전체 v5 재학습 (~10시간, ~100 CU)')

print(f'\n실행 시작: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Strategy : {CONFIG_V5["strategy"]}  |  label_smoothing={CONFIG_V5["label_smoothing"]}')
print(f'Device   : {DEVICE}')
print('='*60)

t0 = time.time()
LOSO_RESULTS_V5 = run_loso_v5(target_datasets, CONFIG_V5, DEVICE)
elapsed = time.time() - t0

print(f'\n🏁 v5 LOSO 완료 — 소요시간: {elapsed/60:.1f}분')
if ABLATION_ONLY:
    print(f'   ※ 타겟 ablation 결과 — 전체 비교는 8-D 참고')


In [ ]:
# ── STEP 8-D: v4 vs v5 성능 비교 ─────────────────────────────────
# ABLATION_ONLY=True인 경우 → 편향 10명 구간에서만 비교
# ABLATION_ONLY=False인 경우 → 전체 52명 비교

df_v4 = pd.DataFrame(LOSO_RESULTS)
df_v5 = pd.DataFrame(LOSO_RESULTS_V5)

ok_v4_all = df_v4[df_v4['status'].isin(['OK', 'RESUMED'])].copy()
ok_v5     = df_v5[df_v5['status'].isin(['OK', 'RESUMED'])].copy()

# v4에서 v5와 같은 SID만 추출해 공정 비교
common_sids = set(ok_v5['sid'].astype(int)) & set(ok_v4_all['sid'].astype(int))
ok_v4 = ok_v4_all[ok_v4_all['sid'].astype(int).isin(common_sids)].copy()

ablation_tag = f'편향 {len(common_sids)}명 (ablation)' if ABLATION_ONLY else f'전체 {len(common_sids)}명'

print('='*65)
print(f'  baseline_v4 vs baseline_v5 비교 — {ablation_tag}')
print(f'  label_smoothing: 없음 → {CONFIG_V5["label_smoothing"]}')
print('='*65)
print(f'  {"지표":<15}  {"v4 (vanilla)":>18}  {"v5 (smoothing)":>18}  {"Δ":>8}')
print('  ' + '-'*63)

for col, label in [
    ('acc',      'Accuracy      '),
    ('f1_macro', 'F1-macro      '),
    ('kappa',    "Cohen's κ     "),
    ('itr',      'ITR (bits/min)'),
]:
    m4, s4 = ok_v4[col].mean(), ok_v4[col].std()
    m5, s5 = ok_v5[col].mean(), ok_v5[col].std()
    delta = m5 - m4
    sign  = '↑' if delta > 0 else ('↓' if delta < 0 else '─')
    print(f'  {label}  {m4:.4f}±{s4:.4f}  {m5:.4f}±{s5:.4f}  {sign}{abs(delta):+.4f}')

# 클래스 편향 개선 분석 (핵심 결과)
print('\n── Right MI 편향 개선 현황 ─────────────────────────────────')
def get_biased(ok_df):
    biased = []
    for _, row in ok_df.iterrows():
        try:
            cm = np.array(ast.literal_eval(str(row['cm'])))
            if cm.shape != (2, 2): continue
            l_rec = cm[0,0]/cm[0].sum() if cm[0].sum() > 0 else 0
            r_rec = cm[1,1]/cm[1].sum() if cm[1].sum() > 0 else 0
            if abs(l_rec - r_rec) > 0.3:
                biased.append({'sid': int(row['sid']), 'l': l_rec, 'r': r_rec})
        except Exception:
            pass
    return biased

biased_v4 = get_biased(ok_v4)
biased_v5 = get_biased(ok_v5)
biased_v4_sids = {b['sid'] for b in biased_v4}
biased_v5_sids = {b['sid'] for b in biased_v5}
resolved = biased_v4_sids - biased_v5_sids
still    = biased_v4_sids & biased_v5_sids
new_bias = biased_v5_sids - biased_v4_sids

print(f'  v4 편향 피험자 ({len(biased_v4)}명): {sorted(biased_v4_sids)}')
print(f'  v5 편향 피험자 ({len(biased_v5)}명): {sorted(biased_v5_sids)}')
print(f'  ✅ 편향 해소: {len(resolved)}명 → {sorted(resolved)}')
if still:
    print(f'  ⚠️  여전히 편향: {len(still)}명 → {sorted(still)}')
if new_bias:
    print(f'  🔴 신규 편향 (v5 부작용): {len(new_bias)}명 → {sorted(new_bias)}')

# per-subject recall 변화 출력
print('\n  [피험자별 Left/Right Recall 변화]')
print(f'  {"sid":>4}  {"v4 Left":>8}  {"v4 Right":>9}  {"v5 Left":>8}  {"v5 Right":>9}  판정')
print('  ' + '-'*60)
for sid in sorted(common_sids):
    r4 = ok_v4[ok_v4['sid'].astype(int) == sid].iloc[0]
    r5 = ok_v5[ok_v5['sid'].astype(int) == sid].iloc[0]
    try:
        cm4 = np.array(ast.literal_eval(str(r4['cm'])))
        cm5 = np.array(ast.literal_eval(str(r5['cm'])))
        l4 = cm4[0,0]/cm4[0].sum(); rr4 = cm4[1,1]/cm4[1].sum()
        l5 = cm5[0,0]/cm5[0].sum(); rr5 = cm5[1,1]/cm5[1].sum()
        still_b = abs(l5-rr5) > 0.3
        verdict = '⚠️  편향 지속' if still_b else '✅ 개선'
        print(f'  s{sid:02d}   {l4:.2f}      {rr4:.2f}       {l5:.2f}      {rr5:.2f}      {verdict}')
    except Exception:
        pass

# CSV 저장
ts = datetime.now().strftime('%Y%m%d_%H%M')
cmp_path = os.path.join(CONFIG['results_dir'],
    f'compare_v4_v5_member_{CONFIG["member"]}_{ts}.csv')
rows = []
for sid in sorted(common_sids):
    r4 = ok_v4[ok_v4['sid'].astype(int) == sid].iloc[0]
    r5 = ok_v5[ok_v5['sid'].astype(int) == sid].iloc[0]
    rows.append({'sid': sid,
                 'v4_acc': r4['acc'], 'v5_acc': r5['acc'],
                 'v4_f1': r4['f1_macro'], 'v5_f1': r5['f1_macro'],
                 'v4_kappa': r4['kappa'], 'v5_kappa': r5['kappa'],
                 'v4_biased': sid in biased_v4_sids,
                 'v5_biased': sid in biased_v5_sids})
pd.DataFrame(rows).to_csv(cmp_path, index=False)

# ── 시각화 ───────────────────────────────────────────────────────
n_sub = len(common_sids)
sids_sorted = sorted(common_sids)
x = range(n_sub)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'v4 vs v5 비교 ({ablation_tag}, label_smoothing={CONFIG_V5["label_smoothing"]})',
             fontsize=12, fontweight='bold')

for ax, (col, title) in zip(axes[:2], [('acc','Accuracy'), ('f1_macro','F1-macro')]):
    v4_vals = [ok_v4[ok_v4['sid'].astype(int)==s].iloc[0][col] for s in sids_sorted]
    v5_vals = [ok_v5[ok_v5['sid'].astype(int)==s].iloc[0][col] for s in sids_sorted]
    ax.plot(x, v4_vals, 'o--', color='#aec7e8', label='v4', linewidth=1.5)
    ax.plot(x, v5_vals, 's-',  color='#2ca02c', label='v5', linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels([f's{s:02d}' for s in sids_sorted], rotation=90, fontsize=8)
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

# Left vs Right Recall 비교 (핵심 플롯)
ax = axes[2]
l_recs_v4, r_recs_v4, l_recs_v5, r_recs_v5 = [], [], [], []
for sid in sids_sorted:
    for ok_df, l_list, r_list in [(ok_v4, l_recs_v4, r_recs_v4),
                                   (ok_v5, l_recs_v5, r_recs_v5)]:
        row = ok_df[ok_df['sid'].astype(int)==sid].iloc[0]
        try:
            cm = np.array(ast.literal_eval(str(row['cm'])))
            l_list.append(cm[0,0]/cm[0].sum())
            r_list.append(cm[1,1]/cm[1].sum())
        except Exception:
            l_list.append(0); r_list.append(0)

w = 0.2
xi = np.array(list(x))
ax.bar(xi-0.3, l_recs_v4, w, label='v4 Left',  color='#aec7e8')
ax.bar(xi-0.1, r_recs_v4, w, label='v4 Right', color='#1f77b4')
ax.bar(xi+0.1, l_recs_v5, w, label='v5 Left',  color='#98df8a')
ax.bar(xi+0.3, r_recs_v5, w, label='v5 Right', color='#2ca02c')
ax.set_xticks(xi)
ax.set_xticklabels([f's{s:02d}' for s in sids_sorted], rotation=90, fontsize=8)
ax.set_title('Left vs Right Recall 변화\n(v5에서 Left↑ 이면 편향 개선)')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
cmp_fig_path = os.path.join(CONFIG['results_dir'],
    f'compare_v4_v5_member_{CONFIG["member"]}_{ts}.png')
plt.savefig(cmp_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ 비교 CSV: {cmp_path}')
print(f'✅ 비교 그래프: {cmp_fig_path}')


## STEP 8-E: Label Smoothing ε Sweep (0.03 / 0.05 / 0.07)

**목표**: ε=0.1이 과도한 보정(성능 하락)으로 확인됨 → 최적 ε 탐색

| ε | 예상 효과 |
|---|---------|
| 0.03 | 약한 정규화, 성능 손실 최소 |
| **0.05** | **적당한 편향-성능 균형 (권장)** |
| 0.07 | ε=0.1보다 보수적, 편향 해소 강도 중간 |

> ⚠️ **CU 주의**: ε 1개 × 편향 10명 ≈ ~2시간 → ~20 CU  
> 기본값 `SWEEP_EPSILONS = [0.05]` — CU 여유 시 `[0.03, 0.05, 0.07]`로 변경

In [ ]:
SWEEP_EPSILONS = [0.05]  

print(f'🎯 ε sweep 대상: {sorted(target_datasets.keys())} ({len(target_datasets)}명)')
print(f'   sweep ε 값: {SWEEP_EPSILONS}')
print(f'   예상 소요: ~{len(SWEEP_EPSILONS)*2}시간 (A100 기준)\n')

SWEEP_RESULTS = {}   # {eps: [dict, ...]}

for eps in SWEEP_EPSILONS:
    tag     = f'v5_eps{str(eps).replace(".", "")}'
    cfg_eps = copy.deepcopy(CONFIG)
    cfg_eps.update({
        'strategy':        f'baseline_{tag}',
        'label_smoothing': eps,
        'ckpt_dir':        f'{DRIVE_ROOT}/results/checkpoints_A_{tag}',
    })
    os.makedirs(cfg_eps['ckpt_dir'], exist_ok=True)

    print(f'{"="*60}')
    print(f'  ε = {eps}  (전략: {cfg_eps["strategy"]})')
    print(f'{"="*60}')

    t0 = time.time()
    SWEEP_RESULTS[eps] = run_loso_v5(target_datasets, cfg_eps, DEVICE)
    elapsed = time.time() - t0
    print(f'  ✅ ε={eps} 완료 — 소요: {elapsed/60:.1f}분\n')

print(f'🏁 ε sweep 완료: ε={list(SWEEP_RESULTS.keys())}')


In [ ]:
# ── STEP 8-F: Multi-ε 비교 분석 ─────────────────────────────────
# v4(ε=0) · sweep 결과 · v5(ε=0.1) 를 한눈에 비교
# ※ SWEEP_RESULTS에 없는 ε는 자동으로 제외됨

import ast

# ── 데이터 준비 ──────────────────────────────────────────────────
sweep_dfs = {}
for eps, res in SWEEP_RESULTS.items():
    df_tmp = pd.DataFrame(res)
    sweep_dfs[eps] = df_tmp[df_tmp['status'].isin(['OK', 'RESUMED'])].copy()

# 비교에 포함할 (ε, DataFrame) 순서 리스트
# v4(ε=0), sweep ε들, v5(ε=0.1)
ref_v4 = ok_v4_all[ok_v4_all['sid'].astype(int).isin(set(BIASED_SIDS))].copy()
eps_df_pairs = [(0.0, ref_v4)]
for eps in sorted(SWEEP_RESULTS.keys()):
    eps_df_pairs.append((eps, sweep_dfs[eps]))
eps_df_pairs.append((0.1, ok_v5.copy()))

# 공통 SID
common_sweep_sids = set(ref_v4['sid'].astype(int))
for eps, df in eps_df_pairs[1:]:
    common_sweep_sids &= set(df['sid'].astype(int))

eps_df_pairs_common = [
    (eps, df[df['sid'].astype(int).isin(common_sweep_sids)].copy())
    for eps, df in eps_df_pairs
]

print(f'비교 대상: SID {sorted(common_sweep_sids)} ({len(common_sweep_sids)}명)')
print(f'비교 ε 값: {[e for e, _ in eps_df_pairs_common]}\n')

# ── 1. 통계 테이블 ────────────────────────────────────────────────
print(f'{"ε":>6}  {"Acc":>8}  {"F1":>8}  {"κ":>8}  {"편향수":>6}')
print('-'*44)
for eps, df in eps_df_pairs_common:
    biased_n = len(get_biased(df))
    label = f'{eps:.2f}' if isinstance(eps, float) else str(eps)
    suffix = '  ← 기준' if eps == 0.0 else ('  ← v5 기존' if eps == 0.1 else '')
    print(f'{label:>6}  {df["acc"].mean():.4f}  {df["f1_macro"].mean():.4f}'
          f'  {df["kappa"].mean():.4f}  {biased_n:>6}{suffix}')

# ── 2. 트레이드오프 라인 플롯 ─────────────────────────────────────
eps_vals   = [e for e, _ in eps_df_pairs_common]
metrics_info = [('acc', 'Accuracy'), ('f1_macro', 'F1-macro'), ('kappa', "Cohen's κ")]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Label Smoothing ε Sweep — 편향 피험자 성능 트레이드오프', fontsize=13, fontweight='bold')

for ax, (metric, title) in zip(axes, metrics_info):
    means = [df[metric].mean() for _, df in eps_df_pairs_common]
    stds  = [df[metric].std()  for _, df in eps_df_pairs_common]
    ax.errorbar(eps_vals, means, yerr=stds, marker='o', linewidth=2,
                capsize=4, color='steelblue', ecolor='lightblue', label='mean±std')
    best_idx = int(np.argmax(means))
    ax.scatter([eps_vals[best_idx]], [means[best_idx]],
               color='red', zorder=5, s=100, label=f'best ε={eps_vals[best_idx]}')
    ax.axvline(0.0, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(0.1, color='orange', linestyle='--', alpha=0.5)
    ax.set_xlabel('ε (label smoothing)')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
sweep_fig_path = os.path.join(
    CONFIG['results_dir'],
    f'sweep_eps_comparison_{datetime.now().strftime("%Y%m%d_%H%M")}.png'
)
plt.savefig(sweep_fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 그래프 저장: {sweep_fig_path}')

# ── 3. 편향 해소 현황 ─────────────────────────────────────────────
ref_biased_sids = {b['sid'] for b in get_biased(ref_v4)}
print(f'\n── ε별 편향 해소 현황 (기준: v4 편향 {len(ref_biased_sids)}명) ─────────')
print(f'  {"ε":>6}  {"편향수":>6}  {"해소수":>6}  해소 SID')
print('  ' + '-'*50)
for eps, df in eps_df_pairs_common:
    b_sids = {b['sid'] for b in get_biased(df)}
    res_n  = ref_biased_sids - b_sids
    label  = f'{eps:.2f}' if isinstance(eps, float) else str(eps)
    print(f'  {label:>6}  {len(b_sids):>6}  {len(res_n):>6}  {sorted(res_n)}')

# ── 4. CSV 저장 ───────────────────────────────────────────────────
rows_sweep = []
for eps, df in eps_df_pairs_common:
    for sid in sorted(common_sweep_sids):
        r = df[df['sid'].astype(int) == sid]
        if r.empty:
            continue
        r = r.iloc[0]
        try:
            cm_arr = np.array(ast.literal_eval(str(r['cm'])))
            l_rec  = cm_arr[0,0]/cm_arr[0].sum() if cm_arr[0].sum() > 0 else 0
            r_rec  = cm_arr[1,1]/cm_arr[1].sum() if cm_arr[1].sum() > 0 else 0
        except Exception:
            l_rec = r_rec = float('nan')
        rows_sweep.append({
            'eps': eps, 'sid': int(sid),
            'acc': r['acc'], 'f1': r['f1_macro'], 'kappa': r['kappa'],
            'left_recall': l_rec, 'right_recall': r_rec,
            'biased': abs(l_rec - r_rec) > 0.3,
        })

df_sweep_all = pd.DataFrame(rows_sweep)
sweep_csv_path = os.path.join(
    CONFIG['results_dir'],
    f'sweep_eps_detail_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
)
df_sweep_all.to_csv(sweep_csv_path, index=False)
print(f'\n📁 상세 CSV 저장: {sweep_csv_path}')
print('\n✅ STEP 8-F 완료')


## STEP 8-G: Class-Weighted Loss Ablation (v6)

**목표**: label smoothing 대신 **방향성 있는 편향 보정** — Right MI(class 1) 패널티만 낮춤

| 전략 | 식 | 의도 |
|------|----|------|
| v4 (baseline) | `weight=[1.0, 1.0]` | 균등 |
| **v6 (class-weight)** | `weight=[w_L, w_R]`, w_L > w_R | Left MI 오분류 더 중하게 패널티 → 모델이 Left 예측 적극화 |

**Sweep**: `[1.2/0.8, 1.3/0.7, 1.4/0.6]` — 기본값 `[1.3, 0.7]` (CU ~20)

> 차이점: label smoothing은 **양쪽 자신감을 모두 억제**,  
> class-weight는 **Left 오분류 패널티만 높임** → 편향만 타겟

In [ ]:
# ── STEP 8-G-A: run_loso_fold_v6 / run_loso_v6 함수 정의 ─────────
# v5 함수 대비 변경점: label_smoothing → class_weight 적용

def run_loso_fold_v6(test_sid: int, all_datasets: dict, cfg: dict, device: str) -> dict:
    """
    baseline_v6 전용 단일 fold.
    ★ 핵심 변경: CrossEntropyLoss(weight=[w_left, w_right])
      w_left > w_right → Left MI 오분류 패널티 상향 → Right MI 과신 억제
    """
    ckpt_path = os.path.join(cfg['ckpt_dir'], f'best_s{test_sid:02d}.pt')
    test_ds = all_datasets[test_sid]
    test_loader = DataLoader(
        test_ds, batch_size=cfg['batch_size'], shuffle=False,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
    )
    use_amp = cfg.get('use_amp', False) and device != 'cpu'
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ★ 핵심 변경: class weight 적용
    cw = cfg.get('class_weight', [1.0, 1.0])
    weight_tensor = torch.tensor(cw, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weight_tensor)

    # ── 체크포인트 복원 ───────────────────────────────────────────
    if os.path.exists(ckpt_path):
        torch.manual_seed(cfg['random_seed'])
        model = HybridBCIModel(cfg).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        _, best_f1, best_preds, best_trues = eval_epoch(
            model, test_loader, criterion, device, scaler
        )
        metrics = evaluate_predictions(best_trues, best_preds,
                                       n_classes=cfg['n_classes'], trial_sec=4.0)
        return {'sid': test_sid, 'status': 'RESUMED', 'best_epoch': -1, **metrics}

    # ── 신규 학습 ─────────────────────────────────────────────────
    train_sids = [s for s in all_datasets if s != test_sid]
    train_ds   = ConcatDataset([all_datasets[s] for s in train_sids])
    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'], shuffle=True,
        num_workers=cfg['num_workers'], pin_memory=cfg['pin_memory'],
        drop_last=True,
    )

    torch.manual_seed(cfg['random_seed'])
    model     = HybridBCIModel(cfg).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay']
    )

    best_f1, best_preds, best_trues, no_improve = -1.0, None, None, 0

    for epoch in range(1, cfg['n_epochs'] + 1):
        train_epoch(model, train_loader, optimizer, criterion, device, scaler)
        _, val_f1, preds, trues = eval_epoch(
            model, test_loader, criterion, device, scaler
        )
        if val_f1 > best_f1:
            best_f1, best_preds, best_trues, no_improve = val_f1, preds.copy(), trues.copy(), 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            no_improve += 1

        if no_improve >= cfg['patience']:
            print(f'    s{test_sid:02d} → Early stop ep{epoch} | best F1={best_f1:.4f}')
            break

    metrics = evaluate_predictions(best_trues, best_preds,
                                   n_classes=cfg['n_classes'], trial_sec=4.0)
    return {'sid': test_sid, 'status': 'OK', 'best_epoch': epoch - no_improve, **metrics}


def run_loso_v6(all_datasets: dict, cfg: dict, device: str) -> list:
    """v6 전용 LOSO 루프 (class-weighted loss)."""
    cw = cfg.get('class_weight', [1.0, 1.0])
    progress_csv = os.path.join(
        cfg['results_dir'],
        f'loso_progress_member_{cfg["member"]}_{cfg["strategy"]}.csv'
    )

    if os.path.exists(progress_csv):
        done_df  = pd.read_csv(progress_csv)
        done_ids = set(done_df['sid'].astype(int).tolist())
        results  = done_df.to_dict('records')
        print(f'📂 이전 진행 결과 로드: {len(done_ids)}개 fold 복원')
    else:
        done_ids, results = set(), []

    remaining = [s for s in sorted(all_datasets.keys()) if s not in done_ids]
    print(f'▶ 남은 fold: {len(remaining)}개 / 전체 {len(all_datasets)}개')
    print(f'  class_weight = Left={cw[0]:.1f}, Right={cw[1]:.1f}')

    for test_sid in tqdm(remaining, desc=f'LOSO {cfg["strategy"]}'):
        try:
            r = run_loso_fold_v6(test_sid, all_datasets, cfg, device)
            results.append(r)
            tag = '🔄' if r['status'] == 'RESUMED' else '✅'
            print(f'  {tag} s{test_sid:02d} [{r["status"]}] — '
                  f'acc={r["acc"]:.3f}  F1={r["f1_macro"]:.3f}  '
                  f'κ={r["kappa"]:.3f}  ITR={r["itr"]:.1f} bits/min')
        except Exception:
            traceback.print_exc()
            results.append({'sid': test_sid, 'status': 'ERROR'})
            print(f'  ❌ s{test_sid:02d}: ERROR — 건너뜀')

        pd.DataFrame(results).to_csv(progress_csv, index=False)

    return results


print('✅ STEP 8-G-A 함수 정의 완료')
print('  run_loso_fold_v6 / run_loso_v6')
print('  변경점: CrossEntropyLoss(weight=[w_left, w_right])')


In [ ]:
# ── STEP 8-G-B: class-weight sweep 실행 ──────────────────────────
# 각 weight 조합마다 별도 ckpt_dir + progress CSV
#
# [컴퓨팅 단위 추정]
#   weight 1개 × 편향 10명 ablation: ~2시간 → ~20 CU
#   weight 3개: ~60 CU 필요
#
# [모드 선택]
#   WEIGHT_SWEEP = [[1.3, 0.7]]            ← 권장 (CU 절약, ~20 CU)
#   WEIGHT_SWEEP = [[1.2,0.8],[1.3,0.7],[1.4,0.6]]  ← 전체 sweep (~60 CU)

WEIGHT_SWEEP = [[1.3, 0.7]]   # ← 필요 시 변경

print(f'🎯 class-weight sweep 대상: {sorted(target_datasets.keys())} ({len(target_datasets)}명)')
print(f'   sweep weight 조합: {WEIGHT_SWEEP}')
print(f'   예상 소요: ~{len(WEIGHT_SWEEP)*2}시간 (A100 기준)\n')

CW_SWEEP_RESULTS = {}   # {(w_l, w_r): [dict, ...]}

for cw in WEIGHT_SWEEP:
    w_l, w_r = cw
    tag     = f'v6_w{str(w_l).replace(".", "")}_{str(w_r).replace(".", "")}'
    cfg_cw  = copy.deepcopy(CONFIG)
    cfg_cw.update({
        'strategy':     f'baseline_{tag}',
        'class_weight': cw,
        'ckpt_dir':     f'{DRIVE_ROOT}/results/checkpoints_A_{tag}',
    })
    os.makedirs(cfg_cw['ckpt_dir'], exist_ok=True)

    print(f'{"="*60}')
    print(f'  class_weight = Left={w_l:.1f} / Right={w_r:.1f}  (전략: {cfg_cw["strategy"]})')
    print(f'{"="*60}')

    t0 = time.time()
    CW_SWEEP_RESULTS[tuple(cw)] = run_loso_v6(target_datasets, cfg_cw, DEVICE)
    elapsed = time.time() - t0
    print(f'  ✅ w=[{w_l},{w_r}] 완료 — 소요: {elapsed/60:.1f}분\n')

print(f'🏁 class-weight sweep 완료: {list(CW_SWEEP_RESULTS.keys())}')


In [ ]:
# ── STEP 8-G-C: v4 vs v6 비교 분석 ──────────────────────────────

# ── 데이터 준비 ──────────────────────────────────────────────────
cw_dfs = {}
for cw_key, res in CW_SWEEP_RESULTS.items():
    df_tmp = pd.DataFrame(res)
    cw_dfs[cw_key] = df_tmp[df_tmp['status'].isin(['OK', 'RESUMED'])].copy()

# 비교 리스트: (label, DataFrame)
cw_compare_pairs = [('v4 (no weight)', ref_v4.copy())]
for cw_key in sorted(CW_SWEEP_RESULTS.keys()):
    cw_compare_pairs.append((f'v6 L={cw_key[0]}/R={cw_key[1]}', cw_dfs[cw_key]))
# v5(ε=0.1) 도 같이 표시
cw_compare_pairs.append(('v5 ε=0.10', ok_v5.copy()))

# 공통 SID
common_cw_sids = set(ref_v4['sid'].astype(int))
for _, df in cw_compare_pairs[1:]:
    common_cw_sids &= set(df['sid'].astype(int))

cw_compare_pairs = [
    (lbl, df[df['sid'].astype(int).isin(common_cw_sids)].copy())
    for lbl, df in cw_compare_pairs
]

print(f'비교 대상: SID {sorted(common_cw_sids)} ({len(common_cw_sids)}명)\n')

# ── 1. 통계 테이블 ────────────────────────────────────────────────
print(f'{"전략":<22}  {"Acc":>7}  {"F1":>7}  {"κ":>7}  {"편향수":>6}  {"해소수":>6}')
print('-'*60)
ref_biased_cw = {b['sid'] for b in get_biased(cw_compare_pairs[0][1])}
for lbl, df in cw_compare_pairs:
    b_sids = {b['sid'] for b in get_biased(df)}
    res_n  = ref_biased_cw - b_sids
    marker = ' ← 기준' if lbl.startswith('v4') else ''
    print(f'{lbl:<22}  {df["acc"].mean():.4f}  {df["f1_macro"].mean():.4f}'
          f'  {df["kappa"].mean():.4f}  {len(b_sids):>6}  {len(res_n):>6}{marker}')

# ── 2. per-subject 상세 ───────────────────────────────────────────
print('\n── per-subject Left/Right Recall 변화 ──────────────────────')
print(f'{"sid":>4}', end='')
for lbl, _ in cw_compare_pairs:
    short = lbl.replace('v4 (no weight)', 'v4').replace('v6 ', '').replace('v5 ', 'v5-')
    print(f'  {short[:12]:>14}', end='')
print()
print('-' * (6 + 16 * len(cw_compare_pairs)))

for sid in sorted(common_cw_sids):
    print(f'  s{sid:02d}', end='')
    for lbl, df in cw_compare_pairs:
        row = df[df['sid'].astype(int) == sid]
        if row.empty:
            print(f'  {"N/A":>14}', end='')
            continue
        row = row.iloc[0]
        try:
            cm_arr = np.array(ast.literal_eval(str(row['cm'])))
            l_r = cm_arr[0,0]/cm_arr[0].sum() if cm_arr[0].sum() > 0 else 0
            r_r = cm_arr[1,1]/cm_arr[1].sum() if cm_arr[1].sum() > 0 else 0
            flag = '⚠' if abs(l_r - r_r) > 0.3 else '✓'
            print(f'  {flag}L{l_r:.2f}/R{r_r:.2f}  ', end='')
        except Exception:
            print(f'  {"err":>14}', end='')
    print()

# ── 3. 최종 권고 ─────────────────────────────────────────────────
print('\n── 최종 권고 ────────────────────────────────────────────────')
best_lbl, best_df = max(
    [(lbl, df) for lbl, df in cw_compare_pairs if not lbl.startswith('v4')],
    key=lambda x: x[1]['kappa'].mean()
)
ref_kappa = cw_compare_pairs[0][1]['kappa'].mean()
best_kappa = best_df['kappa'].mean()
delta_k = best_kappa - ref_kappa
best_bias = len({b['sid'] for b in get_biased(best_df)})
print(f'  최고 κ 전략: {best_lbl}')
print(f'  κ: {ref_kappa:.4f} (v4) → {best_kappa:.4f}  (Δ={delta_k:+.4f})')
print(f'  편향 해소: {len(ref_biased_cw)}명 → {best_bias}명 남음')

if delta_k > -0.03 and best_bias < len(ref_biased_cw):
    print('  ✅ 성능 손실 ≤3% + 편향 해소 → 논문 적용 권장')
elif delta_k > -0.07:
    print('  △ 성능 손실 허용 범위 내, 편향 해소 확인 후 결정')
else:
    print('  ❌ 성능 하락 과도 → v4 그대로 + 한계 기술 권장')

# ── 4. CSV 저장 ───────────────────────────────────────────────────
rows_cw = []
for lbl, df in cw_compare_pairs:
    for sid in sorted(common_cw_sids):
        r = df[df['sid'].astype(int) == sid]
        if r.empty: continue
        r = r.iloc[0]
        try:
            cm_arr = np.array(ast.literal_eval(str(r['cm'])))
            l_rec  = cm_arr[0,0]/cm_arr[0].sum() if cm_arr[0].sum() > 0 else 0
            r_rec  = cm_arr[1,1]/cm_arr[1].sum() if cm_arr[1].sum() > 0 else 0
        except Exception:
            l_rec = r_rec = float('nan')
        rows_cw.append({
            'strategy': lbl, 'sid': int(sid),
            'acc': r['acc'], 'f1': r['f1_macro'], 'kappa': r['kappa'],
            'left_recall': l_rec, 'right_recall': r_rec,
            'biased': abs(l_rec - r_rec) > 0.3,
        })

df_cw_all = pd.DataFrame(rows_cw)
cw_csv_path = os.path.join(
    CONFIG['results_dir'],
    f'cw_comparison_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
)
df_cw_all.to_csv(cw_csv_path, index=False)
print(f'\n📁 상세 CSV 저장: {cw_csv_path}')
print('✅ STEP 8-G-C 완료')


## STEP 8-G-D: 전체 class-weight sweep [1.2/0.8, 1.3/0.7, 1.4/0.6] 비교

- [1.3, 0.7] 결과는 이미 `CW_SWEEP_RESULTS`에 캐시됨 → 스킵
- [1.2, 0.8]: 보수적 보정 (s11 역편향 방지 목적)
- [1.4, 0.6]: 공격적 보정 (s15, s36 편향 잔존 해소 목적)
- 실행 후 STEP 8-G-C를 재실행하면 3-way 비교 자동 업데이트

In [ ]:
# ── STEP 8-G-D: 추가 weight sweep [1.2,0.8] + [1.4,0.6] ────────
# [1.3,0.7] 은 이미 CW_SWEEP_RESULTS에 캐시 → 자동 스킵
# [컴퓨팅 단위] 2개 조합 × 편향 10명: ~40 CU

ADDITIONAL_WEIGHTS = [[1.2, 0.8], [1.4, 0.6]]

for cw in ADDITIONAL_WEIGHTS:
    w_l, w_r  = cw
    cw_key    = tuple(cw)

    if cw_key in CW_SWEEP_RESULTS:
        print(f'  ⏭️  w=[{w_l},{w_r}] 이미 완료 → 스킵')
        continue

    tag    = f'v6_w{str(w_l).replace(".", "")}_{str(w_r).replace(".", "")}'
    cfg_cw = copy.deepcopy(CONFIG)
    cfg_cw.update({
        'strategy':     f'baseline_{tag}',
        'class_weight': cw,
        'ckpt_dir':     f'{DRIVE_ROOT}/results/checkpoints_A_{tag}',
    })
    os.makedirs(cfg_cw['ckpt_dir'], exist_ok=True)

    print(f'{"="*60}')
    print(f'  class_weight = Left={w_l:.1f} / Right={w_r:.1f}  (전략: {cfg_cw["strategy"]})')
    print(f'{"="*60}')

    t0 = time.time()
    CW_SWEEP_RESULTS[cw_key] = run_loso_v6(target_datasets, cfg_cw, DEVICE)
    elapsed = time.time() - t0
    print(f'  ✅ w=[{w_l},{w_r}] 완료 — 소요: {elapsed/60:.1f}분\n')

print(f'\n🏁 전체 class-weight sweep 현황: {sorted(CW_SWEEP_RESULTS.keys())}')
print('   → 아래 STEP 8-G-C 셀을 재실행하면 3-way 비교 업데이트됩니다.')
